# Lab 2: Retrieval-Augmented Generation (RAG)

Objective: Build a minimal RAG pipeline using `transformers` with an encoder-only embedder (BERT) and a decoder-only generator (gemma 1B). We focus on understanding internal mechanisms, not efficiency.

This lab includes:
- A brief RAG overview
- Indexing Padua documents with BERT embeddings (token-based chunking)
- Retrieving top-n similar chunks by cosine similarity
- Generating short answers with gemma using retrieved context
- Exercises matching course goals

<style>
img[src="images/image.png"] {
    width: 30%,
    height: 20%;
}
</style>

![alt text](https://github.com/ferrazzipietro/nlp-2026-labs-dei/blob/main/lab3/notebooks/image.png?raw=1)

_img from https://drjulija.github.io/posts/basic-rag/_

## Setup and Imports
We reuse patterns from Lab 1: local `cache_dir`, simple pooling for embeddings, and an optional gemma generation flag.

In [ ]:
import os, json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

print('Transformers version:', __import__('transformers').__version__)
print('Torch version:', torch.__version__)

BASE_DIR = os.path.join('lab2')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# Model IDs (adjust if needed)
BERT_ID = 'bert-base-uncased'
GEMMA_ID = 'unsloth/gemma-3-1B-it'

Transformers version: 5.0.0
Torch version: 2.10.0+cu128


In [ ]:
import os

# Create the data directory if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

# Create the exercise subdirectory if it doesn't exist
EXERCISE_DATA_DIR = os.path.join(DATA_DIR, 'excercise')
os.makedirs(EXERCISE_DATA_DIR, exist_ok=True)

In [ ]:
print(f"Created directory: {DATA_DIR}")
print(f"Created directory: {EXERCISE_DATA_DIR}")

Created directory: lab2/data
Created directory: lab2/data/excercise


Downloading files using `curl`:

In [ ]:
# curl from github
!curl

curl: try 'curl --help' or 'curl --manual' for more information


In [ ]:
from IPython.lib.display import exists
import os

# BASE_DIR, DATA_DIR, EXERCISE_DATA_DIR are already defined.
# For curl, we need to access the absolute paths.
abs_data_dir = os.path.abspath(DATA_DIR)
abs_exercise_data_dir = os.path.abspath(f'{DATA_DIR}/exercise')

# Base URL for the raw GitHub content
GITHUB_RAW_BASE_URL = 'https://raw.githubusercontent.com/ferrazzipietro/nlp-2026-labs-dei/main/lab3/data/'

# Files to download and their local paths relative to the Colab environment
files_to_download_curl = {
    'kb_docs.json': os.path.join(abs_data_dir, 'kb_docs.json'),
    'queries.json': os.path.join(abs_data_dir, 'queries.json'),
    'excercise/docs.txt': os.path.join(abs_exercise_data_dir, 'docs.txt'),
    'excercise/queries.json': os.path.join(abs_exercise_data_dir, 'queries.json')
}
os.makedirs(abs_data_dir, exist_ok=True)
os.makedirs(abs_exercise_data_dir, exist_ok=True)

for github_path, local_path in files_to_download_curl.items():
    url = GITHUB_RAW_BASE_URL + github_path
    print(f'Downloading {url} to {local_path}...')
    # Use f-string for shell command to correctly embed paths with spaces/special characters if any
    !curl -sSL "{url}" -o "{local_path}"
    if os.path.exists(local_path):
        print(f'Successfully downloaded {os.path.basename(local_path)}')
    else:
        print(f'Failed to download {os.path.basename(local_path)}')

print('\nAll files processed.')

Successfully downloaded kb_docs.json
Successfully downloaded queries.json
Successfully downloaded docs.txt
Successfully downloaded queries.json

All files processed.


## Load Embedder (BERT) and Tokenizer
We use BERT as an encoder-only embedder. Embeddings are computed via mean pooling of the last hidden states across tokens.

In [ ]:
tokenizer_bert = AutoTokenizer.from_pretrained(BERT_ID, cache_dir=CACHE_DIR)
model_bert = AutoModel.from_pretrained(BERT_ID, cache_dir=CACHE_DIR)
print('BERT special tokens:', tokenizer_bert.special_tokens_map)
print('BERT dtype:', next(model_bert.parameters()).dtype)
print('BERT device:', next(model_bert.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
BERT dtype: torch.float32
BERT device: cpu


## Load Generator (GEMMA)
We attempt to load a 1B chat model. If the primary model is unavailable, we fall back to Tinygemma 1.1B. Generation is optional and requires resources (GPU recommended).

In [ ]:
def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=CACHE_DIR)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, cache_dir=CACHE_DIR, torch_dtype=torch.float16, device_map='auto'
    )
    return tok, mdl

tokenizer_gemma, model_gemma = load_model(GEMMA_ID)

print('gemma model:', GEMMA_ID)
print('Has chat template?', bool(getattr(tokenizer_gemma, 'chat_template', None)))

config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

gemma model: unsloth/gemma-3-1B-it
Has chat template? True


## Load Documents, Embed them and save to Knowledge Base
We index 25 Padua documents.

In [ ]:
docs_path = os.path.join(DATA_DIR, 'kb_docs.json')
with open(docs_path, 'r', encoding='utf-8') as f:
    DOCS = json.load(f)
print('there are', len(DOCS), 'docs')
print("Title of first doc:", DOCS[0]['title'])

there are 31 docs
Title of first doc: Prato della Valle


In [ ]:
# Build KB chunks
KB_CHUNKS = []
for d in DOCS:
    # Get the code.
    text_to_encode = d['text']
    # Encode the text.
    encodeing_input = tokenizer_bert(text_to_encode, return_tensors='pt', truncation=False, add_special_tokens=False)
    # Get the last hidden state vector.
    hidden_states = model_bert(**encodeing_input).last_hidden_state
    # The embedding is the avg of all the tokens in our input.
    embedding = hidden_states.mean(dim=1).squeeze().detach().cpu().numpy()
    # We create a dict for each text with some metadata + embedding.
    KB_CHUNKS.append({
        'doc_id': d['id'],
        'title': d['title'],
        'text': text_to_encode,
        'token_count': len(text_to_encode.split()),
        'embedding': embedding.tolist()
    })

# Finally we store create our knoledge base to a file.
index_path = os.path.join(DATA_DIR, 'kb_index.json')
with open(index_path, 'w', encoding='utf-8') as f:
    json.dump(KB_CHUNKS, f, ensure_ascii=False, indent=2)

print()
len(KB_CHUNKS), KB_CHUNKS[0]['title']

(31, 'Prato della Valle')

## Retrieval: Top-n Similar Chunks
We compute cosine similarity between the query embedding and KB chunk embeddings and return the top-n chunks.

In [ ]:
# We will use it to compare with our KB.
def embed_query(query):
    enc = tokenizer_bert([query], return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        out = model_bert(**enc) # Get emeddings.
        # Sentence embeddings.
        emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
    return emb.tolist()[0]

# Similarity between our query and KB.
def cosine_similarity_matrix(query, kb_emb):
    query = np.array(query).reshape(1, -1)
    kb_emb = np.array(kb_emb)
    query = query / np.clip(np.linalg.norm(query, axis=1, keepdims=True), 1e-12, None)
    kb_emb = kb_emb / np.clip(np.linalg.norm(kb_emb, axis=1, keepdims=True), 1e-12, None)
    return query @ kb_emb.T

# Get the top documents: the most similar ones.
def retrieve_top_n(query, kb_emb, n=3):
    q = embed_query(query)
    document_embeddings = [chunk['embedding'] for chunk in kb_emb]
    sims = cosine_similarity_matrix(q, document_embeddings)[0]
    top_idx = np.argsort(-sims)[:n]
    return [{
        'text': kb_emb[i]['text'],
        'similarity': float(sims[i])}
        for i in top_idx]

In [ ]:
# Load queries
queries_path = os.path.join(DATA_DIR, 'queries.json')
with open(queries_path, 'r', encoding='utf-8') as f:
    QUERIES = json.load(f)


# Demo retrieval on first query
q0 = QUERIES[0]['query']
print('Query:', q0)
top_chunks = retrieve_top_n(q0, KB_CHUNKS, n=3)
for retrieved in top_chunks:
    print(f'Similarity={retrieved["similarity"]:.3f}, text={retrieved["text"][:180]}')

Query: Where can you see Giotto’s frescoes in Padua?
Similarity=0.653, text=Donatello's statue located in front of the Basilica of Saint Anthony is a masterpiece of Renaissance sculpture. It is now under restoration, but its historical significance and art
Similarity=0.621, text=Founded in 1222, the University of Padua is renowned for research excellence, diverse faculties, and a vibrant international community.
Similarity=0.618, text=The Scrovegni Chapel contains Giotto’s famous frescoes. Entry is by ticket and timed slots, located near the Church of the Eremitani.


It does not seem very good. Let's see what happens with **models explicitly trained on this retrieval task**: `Qwen/Qwen3-Embedding-0.6B`

If you are interested in learning more about better embedders for RAG tasks (and others) look [here](https://huggingface.co/spaces/mteb/leaderboard)

For simplicity, we use a new library (`sentence_transformers`). If you prefer to use `transformers` (it give smore control on every step) you can look [here](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B#transformers-usage)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

# Build KB chunks
KB_CHUNKS = []
for d in DOCS:
    # Get the text.
    text_to_encode = d['text']
    # Get embedding.
    embedding = embedder.encode([text_to_encode])[0]
    # Store to dict and append.
    KB_CHUNKS.append({
        'doc_id': d['id'],
        'title': d['title'],
        'text': text_to_encode,
        'token_count': len(text_to_encode.split()),
        'embedding': embedding.tolist()
    })

# Store KB to disk.
index_path = os.path.join(DATA_DIR, 'kb_index.json')
with open(index_path, 'w', encoding='utf-8') as f:
    json.dump(KB_CHUNKS, f, ensure_ascii=False, indent=2)

def embed_query(query):
    return embedder.encode([query])



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Let's have a look at the same example as before and see if it is better:

In [ ]:
print('Query:', q0)
top_chunks = retrieve_top_n(q0, KB_CHUNKS, n=3)
for retrieved in top_chunks:
    print(f'Similarity={retrieved["similarity"]:.3f}, text={retrieved["text"][:180]}')

Query: Where can you see Giotto’s frescoes in Padua?
Similarity=0.634, text=The Scrovegni Chapel contains Giotto’s famous frescoes. Entry is by ticket and timed slots, located near the Church of the Eremitani.
Similarity=0.593, text=Padua Cathedral (Duomo) stands alongside a Romanesque Baptistery with frescoes by Giusto de’ Menabuoi. Regular liturgical celebrations take place.
Similarity=0.569, text=The Basilica of Saint Anthony in Padua houses Saint Anthony’s tomb and is a major pilgrimage site with daily religious services.


## Generation: Answer from Retrieved Context
We instruct GEMMA to answer concisely using only the provided context. If the answer is not in the context, the model should say it does not know.

In [ ]:
system_prompt = """You are a helpful assistant that answers questions about the city of Padova, Italy.
Use the retrieved documents to answer the question as best as you can.
If you don't know the answer, say you don't know.
Your answer are short and concise"""


RESULTS = []
for query in QUERIES:
    print('Query: ', query['query'])
    top_chunks = retrieve_top_n(query['query'], KB_CHUNKS, n=3)
    for retrieved in top_chunks:
        print(f'Similarity={retrieved["similarity"]:.3f}, text={retrieved["text"][:180]}')
    context = [r['text'] for r in top_chunks]

    user_message = f"""Here is some useful context: {'\n'.join(context)}\n\n Here is the query: {query['query']}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    inputs = tokenizer_gemma.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_gemma.device)

    outputs = model_gemma.generate(**inputs, max_new_tokens=40)
    answer = tokenizer_gemma.decode(outputs[0][inputs["input_ids"].shape[-1]:])
    RESULTS.append({
        'query': query['query'],
        'retrieved': top_chunks,
        'answer': answer
    })
    print()

Query:  Where can you see Giotto’s frescoes in Padua?
Similarity=0.634, text=The Scrovegni Chapel contains Giotto’s famous frescoes. Entry is by ticket and timed slots, located near the Church of the Eremitani.
Similarity=0.593, text=Padua Cathedral (Duomo) stands alongside a Romanesque Baptistery with frescoes by Giusto de’ Menabuoi. Regular liturgical celebrations take place.
Similarity=0.569, text=The Basilica of Saint Anthony in Padua houses Saint Anthony’s tomb and is a major pilgrimage site with daily religious services.

Query:  What is the large oval square with an island called?
Similarity=0.542, text=Prato della Valle is one of Italy's largest squares. It features an oval shape with the Isola Memmia at its center and hosts a lively Saturday market.
Similarity=0.434, text=Palazzo della Ragione is the medieval town hall with a vast wooden-roofed hall. A traditional food market operates below its arcades.
Similarity=0.384, text=Pilgrims visiting the Basilica of Saint Anthony fin

We can look at how the last input to the model looked like:

In [ ]:
print(tokenizer_gemma.decode(inputs['input_ids'], skip_special_tokens=False)[0])

<bos><start_of_turn>user
You are a helpful assistant that answers questions about the city of Padova, Italy.
Use the retrieved documents to answer the question as best as you can.
If you don't know the answer, say you don't know.
Your answer are short and concise

Here is some useful context: Donatello's statue located in front of the Basilica of Saint Anthony is a masterpiece of Renaissance sculpture. It is now under restoration, but its historical significance and artistic value continue to draw attention from art enthusiasts.
The Basilica of Saint Anthony in Padua houses Saint Anthony’s tomb and is a major pilgrimage site with daily religious services.
The Scrovegni Chapel contains Giotto’s famous frescoes. Entry is by ticket and timed slots, located near the Church of the Eremitani.

 Here is the query: Where is the sculpture of Donatello located?<end_of_turn>
<start_of_turn>model



Look at what happens without using RAG

In [ ]:
system_prompt_no_rag = """You are a helpful assistant that answers questions about the city of Padova, Italy.
If you don't know the answer, say you don't know.
Your answer are short and concise"""

for query, answer_rag in zip(QUERIES, RESULTS):
    print('Query: ', query['query'])
    user_message = f"{query['query']}"

    messages = [
        {"role": "system", "content": system_prompt_no_rag},
        {"role": "user", "content": user_message}
    ]

    inputs = tokenizer_gemma.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_gemma.device)

    outputs = model_gemma.generate(**inputs, max_new_tokens=40)
    answer = tokenizer_gemma.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    print('Answer with RAG   : ', answer_rag['answer'])
    print('Answer without RAG: ', answer, '\n')

Query:  Where can you see Giotto’s frescoes in Padua?
Answer with RAG   :  You can see Giotto’s frescoes in the Scrovegni Chapel.<end_of_turn>
Answer without RAG:  The Basilica of Saint Anthony.<end_of_turn> 

Query:  What is the large oval square with an island called?
Answer with RAG   :  Prato della Valle.<end_of_turn>
Answer without RAG:  The Basilica di Sant'Antonio.<end_of_turn> 

Query:  Which basilica houses Saint Anthony’s tomb?
Answer with RAG   :  The Basilica of Saint Anthony in Padua.<end_of_turn>
Answer without RAG:  San Lorenzo Basilica.<end_of_turn> 

Query:  Where should I go to get a gelato?
Answer with RAG   :  Gelateria la Romana.<end_of_turn>
Answer without RAG:  Gelateria del Corso!<end_of_turn> 

Query:  Where is the sculpture of Donatello located?
Answer with RAG   :  The sculpture of Donatello is located in front of the Basilica of Saint Anthony.<end_of_turn>
Answer without RAG:  The sculpture of Donatello is located in the Basilica of San Nicholas.<end_of_turn

## Excercise

1) You have generated an answer for each of the queries in `data/queries.json`. Each query has an `expected_answer`. Find a way to use Gemma to determine whether the generated answer is correct or not by prompting the LLM itself.
We want to check two things:
    * If the answer is consistent wrt context.
    * If the answer is correct wrt to the expected answer (that we have).

2) Create a RAG system that answer the queries in `data/excercise/queries.json` using the documents in `data/excercise/docs.txt`.


# Execrise 1 solution

In [ ]:
# First we try to use the embeddings of anwers and expected
# answers and we comprare them with embeddings.

# Store everything in here.
merged = []
for i, j in zip(QUERIES, RESULTS):
    merged.append(i | j)

# Just store all embedding pairs.
embds = []
for item in merged:
    embds.append(
        [embed_query(item["answer"]), embed_query(item["expected_answer"])]
    )

# Now compute cos similarity for each pair.
import torch
import torch.nn as nn

cos = nn.CosineSimilarity()
for emb, dic in zip(embds, merged):
    emb = torch.from_numpy(emb[0]), torch.from_numpy(emb[1])
    print("Answer:\n", dic["answer"])
    print("Expected Answer:\n", dic["expected_answer"])
    print("Score:", cos(emb[0], emb[1]).item())
    print()

Answer:
 You can see Giotto’s frescoes in the Scrovegni Chapel.<end_of_turn>
Expected Answer:
 Scrovegni Chapel
Score: 0.6759600639343262

Answer:
 Prato della Valle.<end_of_turn>
Expected Answer:
 Prato della Valle
Score: 0.7649486660957336

Answer:
 The Basilica of Saint Anthony in Padua.<end_of_turn>
Expected Answer:
 Basilica of Saint Anthony
Score: 0.8426626920700073

Answer:
 Gelateria la Romana.<end_of_turn>
Expected Answer:
 Gelateria la Romana
Score: 0.8800352811813354

Answer:
 The sculpture of Donatello is located in front of the Basilica of Saint Anthony.<end_of_turn>
Expected Answer:
 Basilica of Saint Anthony square
Score: 0.7306045293807983



In [ ]:
# Now lets try to use the approach suggested by the tutor:
# let the LLM decide.

system_prompt = """You are a system that have to decide if two sentenecs have the same
meaning. You just need to answer with a 'yes' or a 'no'. Don't write anything
else."""


for dic in merged:
    # Get the two sentences.
    s1 = dic["answer"]
    s2 = dic["expected_answer"]

    # Define the user msg.
    user_message = f"Sentence1:'{s1}'\nSentence2:'{s2}'"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    inputs = tokenizer_gemma.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_gemma.device)

    outputs = model_gemma.generate(**inputs, max_new_tokens=40)
    answer = tokenizer_gemma.decode(outputs[0][inputs["input_ids"].shape[-1]:])
    print(answer, "\n")


yes
<end_of_turn> 

yes
<end_of_turn> 

yes
<end_of_turn> 

yes
<end_of_turn> 

yes
<end_of_turn> 

